In [2]:
import pandas as pd

In [3]:
df = pd.DataFrame({
    'Signal': [0,0,1,0,0,1,0,2,2,0,2,1,0,0,1],
    'Returns': [0,0.01,0.24,-0.06,0,0.01,0.032,0.05,-0.021,0.01,0.045,0.025,-0.018,-0.02,0.05]
    })

In [4]:
df

,Signal,Returns
0,0,0.000
1,0,0.010
2,1,0.240
3,0,-0.060
4,0,0.000
5,1,0.010
6,0,0.032
7,2,0.050
8,2,-0.021
9,0,0.010


### **GROUPING** | Calculate for every different type of signals

In [5]:
# we group by a column that contains multiple categories
df.groupby("Signal")

# select a column that you want to analyze
df.groupby("Signal")["Returns"]

# then apply the metric operation
df.groupby("Signal")["Returns"].mean()

Signal
0   -0.005750
1    0.081250
2    0.024667
Name: Returns, dtype: float64

In [6]:
# to get a dataframe:
df.groupby("Signal")["Returns"].mean().reset_index()

,Signal,Returns
0,0,-0.005750
1,1,0.081250
2,2,0.024667


#### Rename using .rename()

_inside:_  
```python
columns = {"old_col_name":"new_col_name", ... , inplace = True}
```

In [7]:
# To get multiple calcs 
summary_df = df.groupby("Signal").agg(
    Count = ("Returns","count"),
    Expected_return = ("Returns","mean")
).reset_index()

summary_df.rename(columns={"Expected_return": "Expected return"}, inplace=True)

summary_df

,Signal,Count,Expected return
0,0,8,-0.005750
1,1,4,0.081250
2,2,3,0.024667


### **P(Return > 0)** for all signals
P(Return > 0 | Signal = 0)  
P(Return > 0 | Signal = 1)  
P(Return > 0 | Signal = 2)

In [8]:
signal_df = df[df["Signal"] == 0]
(signal_df["Returns"]>0).mean()

#then repeat for all other signals

0.375

In [9]:
# Using groupby
# 1st group by the column with category then target the column for calculations
df.groupby("Signal")["Returns"]

# then apply boolean op
df.groupby("Signal")["Returns"].apply(lambda x: (x>0))


Signal    
0       0     False
        1      True
        3     False
        4     False
        6      True
        9      True
        12    False
        13    False
1       2      True
        5      True
        11     True
        14     True
2       7      True
        8     False
        10     True
Name: Returns, dtype: bool

x = Return column for one group

Convert to True/False

In [10]:
# then take average
df.groupby("Signal")["Returns"].apply(lambda x: (x>0).mean())

Signal
0    0.375000
1    1.000000
2    0.666667
Name: Returns, dtype: float64

x = Return column for one group

Convert to True/False

Take average

In [11]:
# convert to df with right column
df.groupby("Signal")["Returns"].apply(lambda x: (x>0).mean()).reset_index(name="P(Returns)>0")

,Signal,P(Returns)>0
0,0,0.375000
1,1,1.000000
2,2,0.666667


### **Expected Returns** and **Volatility** of each Signal

In [12]:
df.groupby("Signal")["Returns"].mean()

Signal
0   -0.005750
1    0.081250
2    0.024667
Name: Returns, dtype: float64

In [13]:
df.groupby("Signal")["Returns"].std()

Signal
0    0.027473
1    0.107112
2    0.039627
Name: Returns, dtype: float64

### **Summary dataframe**
using **.agg()**  

_then inside:_  
```python
col_name = (column, operation)
```

In [14]:
df.groupby("Signal").agg(
    Returns_count = ("Returns", "count"),
    P_positive_returns = ("Returns", lambda x: (x>0).mean()),
    Expected_mean = ("Returns", "mean"),
    Volatility = ("Returns", "std")
)

,Returns_count,P_positive_returns,Expected_mean,Volatility
Signal,,,,
0,8,0.375000,-0.005750,0.027473
1,4,1.000000,0.081250,0.107112
2,3,0.666667,0.024667,0.039627


In [15]:
summary_df = df.groupby("Signal").agg(
    Returns_count = ("Returns", "count"),
    P_positive_returns = ("Returns", lambda x: (x>0).mean()),
    Expected_mean = ("Returns", "mean"),
    Volatility = ("Returns", "std")
)

summary_df.rename(columns={"Expected_mean":"Expected Returns", "Returns_count": "Returns count", "P_positive_returns":"P(Returns>0)"}, inplace=True)
summary_df

,Returns count,P(Returns>0),Expected Returns,Volatility
Signal,,,,
0,8,0.375000,-0.005750,0.027473
1,4,1.000000,0.081250,0.107112
2,3,0.666667,0.024667,0.039627


In [16]:
summary_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Returns count     3 non-null      int64  
 1   P(Returns>0)      3 non-null      float64
 2   Expected Returns  3 non-null      float64
 3   Volatility        3 non-null      float64
dtypes: float64(3), int64(1)
memory usage: 120.0 bytes


In [17]:
summary_df.columns

Index(['Returns count', 'P(Returns>0)', 'Expected Returns', 'Volatility'], dtype='object')

In [18]:
# Create a separate display DataFrame.
display_df = summary_df.copy()

pct_cols = [
    "P(Returns>0)",
    "Expected Returns",
    "Volatility"
]

for col in pct_cols:
    display_df[col] = (display_df[col] * 100).round(1).astype(str) + "%"

In [19]:
display_df

,Returns count,P(Returns>0),Expected Returns,Volatility
Signal,,,,
0,8,37.5%,-0.6%,2.7%
1,4,100.0%,8.1%,10.7%
2,3,66.7%,2.5%,4.0%
